# Module 03: EDA

In [ ]:
# packages
import numpy as np 
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
from sklearn.model_selection import train_test_split 
from ISLP import load_data

# set seed
seed = 2323

# We'll use the Hitters data from ISLP for this activity. The metadata for Hitters can be found here:
# https://intro-stat-learning.github.io/ISLP/datasets/Hitters.html


In [ ]:
# Load the data
Hitters = load_data('Hitters')

### Determine the number of rows and columns in the dataset by returning its "shape" attribute

In [ ]:
Hitters.shape

### Determine whether each feature is numeric or categorical by returning the "dtype" attribute for each column

In [ ]:
for col in Hitters.columns:
    print(f"{col}: {Hitters[col].dtype}")

### Before doing any other analyses, let's create training and test sets.

In [ ]:
Train, Test = train_test_split(Hitters, 
                               random_state=seed, 
                               test_size=0.40, 
                               shuffle=True) 

### Based on the metadata, what is the difference between the 6 columns starting with 'C' and the 6 related columns that don't?

The columns that start with C are the player’s career totals. The ones without C are just their stats from the 1986 season.

### On the training set, create pairwise scatterplots for each of these 6 columns with the 'Salary' variable.

In [ ]:
career_cols = ['CAtBat', 'CHits', 'CHmRun', 'CRuns', 'CRBI', 'CWalks']
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i, col in enumerate(career_cols):
    axes[i].scatter(Train[col], Train['Salary'], alpha=0.5)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Salary')
    axes[i].set_title(f'{col} vs Salary')

plt.tight_layout()
plt.show()

### Use the "describe" method to determine the mean, standard deviation, and 5 number summary of all numeric variables in the training subset of _Hitters_.

In [ ]:
Train.describe()

### It looks like the mean and median of 'AtBat' are nearly equal. This _might_ suggest that this variable is normally distributed. Create a histogram of 'AtBat' to check this hypothesis.

In [ ]:
plt.figure(figsize=(8, 6))
plt.hist(Train['AtBat'], bins=20, edgecolor='black')
plt.xlabel('AtBat')
plt.ylabel('Frequency')
plt.title('Histogram of AtBat')
plt.show()

### Let's standardize the AtBat feature (i.e., normalize by z-scores). We'll create a new column in the training data called 'AtBat_st' to represent this.

In [ ]:
Train['AtBat_st'] = (Train['AtBat'] - Train['AtBat'].mean()) / Train['AtBat'].std()

### How many rows have an 'AtBat' value within the first standard deviation?

Hint: the 'len' magic method returns the number of rows of a dataFrame.

In [ ]:
len(Train[(Train['AtBat_st'] >= -1) & (Train['AtBat_st'] <= 1)])

### Going back to the results of the 'describe' method, how can you tell that the 'Salary' variable has missing values?

You can tell because the count for Salary is smaller than the total number of rows.

### Describe a situation where a variable could have missing values but this would not be reflected in the results of the 'describe' method.

One example is if a column is categorical. describe() usually focuses on numeric columns, so you might not see missing values there unless you check it yourself.

### On the training data, create separate boxplots of the 'AtBat' variable for when 'Salary' is populated or missing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

Train[Train['Salary'].notna()].boxplot(column='AtBat', ax=axes[0])
axes[0].set_title('AtBat (Salary Populated)')
axes[0].set_ylabel('AtBat')

Train[Train['Salary'].isna()].boxplot(column='AtBat', ax=axes[1])
axes[1].set_title('AtBat (Salary Missing)')
axes[1].set_ylabel('AtBat')

plt.tight_layout()
plt.show()

### Create a correlation matrix for all numeric features in the training set

In [ ]:
Train.corr(numeric_only=True)

### Propose two different ways of imputing the missing values of Salary while taking advantage of the information given in the boxplots or the correlation matrix.

1# We can use a prediction model. Since Salary has a correlation to CRuns, CHits, CHmRun, and Walks, we could build a regression model using those and use it to predict the missing values.

2# We could group similar profiles. Since the boxplots show missing Salary players tend to have a lower AtBat, we could group players by AtBat range and fill in missing Salary with the median Salary from that group.

### For our last exercise, we'll explore Hits and Walks relative to AtBat totals. 
- Use the sum function to calculuate the totals of each of these three variables for the 1986 season (on the training set). 
- Create a pie chart which shows total hits, total walks, and remaining total (neither) as percents of the At Bats total (on the training set). 

In [ ]:
TotHits = Train['Hits'].sum()
TotWalks = Train['Walks'].sum()
TotAtBat = Train['AtBat'].sum()

Labels = ['Hits', 'Walks', 'Neither']
Totals = [TotHits, TotWalks, TotAtBat - TotHits - TotWalks]

In [ ]:
# pie chart
plt.figure(figsize=(8, 8))
plt.pie(Totals, labels=Labels, autopct='%1.1f%%')
plt.title('Distribution of At Bats (1986 Season)')
plt.show()

### The previous two cells gave us totals across all players. For each player in the training set, calculate the Hits as a percent of AtBat and store it in a new variable called 'AVG'

In [ ]:
Train['AVG'] = Train['Hits'] / Train['AtBat']

### Using 0.25 and 0.31 as the split points, create a new variable with three bins: high, medium, and low. 

In [ ]:
Train['AVG_bin'] = 'medium'
Train.loc[Train['AVG'] < 0.25, 'AVG_bin'] = 'low'
Train.loc[Train['AVG'] > 0.31, 'AVG_bin'] = 'high'

### Create a bar chart that displays the number of players in each of the low, medium, and high categories (for the training data).

In [ ]:
counts = Train['AVG_bin'].value_counts()
indexMap = ['low', 'medium', 'high']
reordered_list = [counts.get(i, 0) for i in indexMap]

plt.bar(range(len(indexMap)), reordered_list)
plt.title('1986 AVG (Training Set)')
plt.ylabel('Number of Players')
plt.xticks(range(len(indexMap)), indexMap)
plt.show()

### Did we use the depth method or width method for creating these bins? Explain.

For this we use the width method. It creates fixed boundaries for the bins. We didn’t try to make each bin have the same number of players, we just set the cutoffs.